# LeetCode 1774 — Closest Dessert Cost bằng Python

Link đề: https://leetcode.com/problems/closest-dessert-cost/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Backtracking

Skeleton LeetCode:

```python
class Solution:
    def closestCost(self, baseCosts: List[int], toppingCosts: List[int], target: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Liệt kê mọi tổ hợp "mỗi topping lấy 0, 1 hoặc 2 phần" bằng `itertools.product` (`3^m` tổ hợp).
2. So sánh "gần nhất, hòa thì lấy nhỏ hơn" bằng **khóa tuple** `(|c − target|, c)`.
3. Viết **DFS quay lui có cắt tỉa**: dừng khi chi phí đã vượt mục tiêu.
4. Viết **quy hoạch động tập tổng đạt được** (knapsack bị chặn) chỉ giữ các tổng `≤ target` và theo dõi tổng nhỏ nhất vượt target.
5. Dùng **meet-in-the-middle** + `bisect` để giảm từ `3^m` xuống `~3^{m/2}`.

## 2. Đề bài & ví dụ

Có `n` loại **kem nền** và `m` loại **topping**. Quy tắc:
- Có **đúng một** kem nền.
- Có thể thêm một hoặc nhiều loại topping, hoặc không thêm gì.
- Mỗi loại topping lấy **tối đa hai** phần.

Cho `baseCosts` (giá kem nền), `toppingCosts` (giá **một** phần topping) và `target`. Trả về tổng chi phí **gần `target` nhất**; nếu có nhiều, trả về giá **nhỏ hơn**.

**Ví dụ 1:** `baseCosts = [1,7]`, `toppingCosts = [3,4]`, `target = 10` → `10` (nền 7 + 1 phần topping 3).

**Ví dụ 2:** `baseCosts = [2,3]`, `toppingCosts = [4,5,100]`, `target = 18` → `17` (3 + 4 + 2·5).

**Ví dụ 3:** `baseCosts = [3,10]`, `toppingCosts = [2,5]`, `target = 9` → `8` (có thể làm 8 và 10, cùng cách 1, lấy 8).

**Ràng buộc:** `1 ≤ n, m ≤ 10`; `1 ≤ baseCosts[i], toppingCosts[i] ≤ 10⁴`; `1 ≤ target ≤ 10⁴`.

In [1]:
import random
import time
import bisect
from itertools import product
from typing import List

examples = [([1, 7], [3, 4], 10, 10), ([2, 3], [4, 5, 100], 18, 17), ([3, 10], [2, 5], 9, 8)]
for b, t, tg, exp in examples:
    print(f"base={b}, topping={t}, target={tg} -> kỳ vọng {exp}")

base=[1, 7], topping=[3, 4], target=10 -> kỳ vọng 10
base=[2, 3], topping=[4, 5, 100], target=18 -> kỳ vọng 17
base=[3, 10], topping=[2, 5], target=9 -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1 `itertools.product` — tích Descartes

`product(range(3), repeat=m)` sinh mọi bộ `(k_0, …, k_{m−1})` với `k_i ∈ {0,1,2}` — đúng `3^m` bộ.

In [2]:
print(list(product(range(3), repeat=2)), 3 ** 10)

[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)] 59049


### 3.2 Khóa so sánh "gần nhất, hòa lấy nhỏ hơn"

`min(costs, key=lambda c: (abs(c - target), c))`: so khoảng cách trước, hòa thì so chính giá trị.

In [3]:
print(min([8, 10, 9.5, 11], key=lambda c: (abs(c - 9), c)), min([8, 10], key=lambda c: (abs(c - 9), c)))

9.5 8


### 3.3 Đệ quy quay lui (DFS)

`dfs(i, cost)`: đã quyết định topping `0..i−1`; thử `0, 1, 2` phần cho topping `i`. Nếu `cost` đã `≥ target`, thêm topping chỉ làm xa hơn ⇒ **cắt tỉa**.

### 3.4 Tập các tổng đạt được (subset-sum)

Bắt đầu `S = {0}`; với mỗi topping giá `t`: `S = {s + k·t : s ∈ S, k ∈ {0,1,2}}`. Giới hạn phần tử `≤ target` để tập không phình to.

In [4]:
S = {0}
for t in [3, 4]:
    S = {s + k * t for s in S for k in range(3)}
print(sorted(S))

[0, 3, 4, 6, 7, 8, 10, 11, 14]


### 3.5 `bisect` — tìm phần tử gần một giá trị trong mảng đã sắp

`i = bisect_left(arr, x)` ⇒ `arr[i]` là phần tử nhỏ nhất `≥ x`, `arr[i−1]` là phần tử lớn nhất `< x`. Hai ứng viên này là gần `x` nhất.

In [5]:
arr = [0, 3, 4, 6, 7, 8, 11, 14]; x = 5
i = bisect.bisect_left(arr, x)
print("i =", i, "ứng viên:", arr[i - 1], arr[i])

i = 3 ứng viên: 4 6


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 3**: `base ∈ {3, 10}`, topping `2` và `5` (mỗi loại 0–2 phần) ⇒ tổng topping có thể là `{0, 2, 4, 5, 7, 9, 10, 12, 14}`.

| nền | các tổng | gần 9 nhất |
|---|---|---|
| 3 | 3, 5, 7, 8, 10, 12, 13, 15, 17 | 8 và 10 (cách 1) |
| 10 | 10, 12, 14, … | 10 (cách 1) |

Hòa giữa 8 và 10 ⇒ lấy **8**.

Không gian: `n · 3^m ≤ 10 · 59049 ≈ 6·10⁵` tổ hợp — đủ nhỏ để **vét cạn**. Các cách sau tối ưu hằng số / tiệm cận cho trường hợp `m` lớn hơn.

In [6]:
base, top, tg, _ = examples[2]
sums = sorted({sum(k * c for k, c in zip(ks, top)) for ks in product(range(3), repeat=len(top))})
print("tổng topping:", sums)
print({b: sorted(b + s for s in sums) for b in base})

tổng topping: [0, 2, 4, 5, 7, 9, 10, 12, 14]
{3: [3, 5, 7, 8, 10, 12, 13, 15, 17], 10: [10, 12, 14, 15, 17, 19, 20, 22, 24]}


## 5. Cách 1 — Brute Force / baseline: `product` liệt kê mọi tổ hợp

In [7]:
def closest_bruteforce(baseCosts, toppingCosts, target):
    best = None
    for b in baseCosts:
        for ks in product(range(3), repeat=len(toppingCosts)):
            c = b + sum(k * t for k, t in zip(ks, toppingCosts))
            if best is None or (abs(c - target), c) < (abs(best - target), best):
                best = c
    return best

for b, t, tg, exp in examples:
    got = closest_bruteforce(b, t, tg); print(got, "kỳ vọng", exp); assert got == exp

10 kỳ vọng 10
17 kỳ vọng 17
8 kỳ vọng 8


**Độ phức tạp:** `O(n · 3^m · m)` ≈ `6·10⁶` phép — chạy được (~1 giây trong Python với `m = 10`).

**Nút thắt:** (1) mỗi tổ hợp tính tổng lại từ đầu (`· m`); (2) tính tổng topping lặp lại cho **mỗi** nền; (3) không cắt tỉa khi đã vượt mục tiêu.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tách nền và topping):** tổng topping không phụ thuộc vào nền ⇒ tính **một lần** tập `T` các tổng topping, rồi với mỗi nền `b` tìm `t ∈ T` sao cho `b + t` gần `target` nhất.

**Insight 2 (cắt tỉa / chặn trên):** chi phí chỉ tăng khi thêm topping. Nếu đã có chi phí `c ≥ target`, mọi mở rộng đều xa hơn ⇒ dừng. Vì vậy chỉ cần biết:
- mọi tổng đạt được `≤ target`;
- tổng **nhỏ nhất** `> target` (ứng viên "vượt" tốt nhất).

**Insight 3 (meet-in-the-middle):** chia topping làm hai nửa, tính tập tổng mỗi nửa (`3^{m/2}` phần tử), sắp nửa sau; với mỗi `(b, s1)` dùng `bisect` tìm `s2` gần `target − b − s1` nhất.

**Quy tắc so sánh:** ứng viên `c` tốt hơn `best` ⇔ `(|c − target|, c) < (|best − target|, best)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS quay lui có cắt tỉa

In [8]:
def closest_dfs(baseCosts, toppingCosts, target):
    best = [min(baseCosts)]
    m = len(toppingCosts)
    def better(c):
        return (abs(c - target), c) < (abs(best[0] - target), best[0])
    def dfs(i, cost):
        if better(cost):
            best[0] = cost
        if i == m or cost >= target:          # hết topping hoặc đã vượt: không thể tốt hơn
            return
        for k in range(3):
            dfs(i + 1, cost + k * toppingCosts[i])
    for b in baseCosts:
        dfs(0, b)
    return best[0]

print([closest_dfs(b, t, tg) for b, t, tg, _ in examples])

[10, 17, 8]


**Code nộp LeetCode (Cách 2):** đổi tên thành `closestCost`. **Độ phức tạp:** tệ nhất `O(n · 3^m)` (khi mọi giá nhỏ so với target), thực tế nhỏ hơn nhiều nhờ cắt tỉa.

### 7.2 Cách 3 — Quy hoạch động: tập tổng `≤ target` + tổng vượt nhỏ nhất

In [9]:
def closest_dp(baseCosts, toppingCosts, target):
    can = [False] * (target + 1)
    over = float("inf")                         # chi phí nhỏ nhất > target
    for b in baseCosts:
        if b <= target: can[b] = True
        else: over = min(over, b)
    for t in toppingCosts:
        prev = can[:]                          # chỉ dùng trạng thái trước topping này
        for s in range(target + 1):
            if not prev[s]:
                continue
            for k in (1, 2):
                v = s + k * t
                if v <= target: can[v] = True
                else:
                    over = min(over, v); break
    below = max((s for s in range(target + 1) if can[s]), default=None)
    if below is None:
        return over
    return below if target - below <= over - target else over

print([closest_dp(b, t, tg) for b, t, tg, _ in examples])

[10, 17, 8]


**Vì sao chỉ cần tổng vượt "một bước":** gọi `c*` là chi phí nhỏ nhất `> target`. Bỏ phần topping cuối cùng (theo thứ tự xử lý) khỏi tổ hợp của `c*` được `c' < c*`; nếu `c' > target` thì `c'` là chi phí vượt nhỏ hơn — mâu thuẫn. Vậy `c' ≤ target` (đã có trong `can`), và `c*` được ghi nhận khi thêm phần topping đó. Tương tự cho trường hợp lấy 2 phần (nếu `s + t > target` thì `s + 2t` không thể là nhỏ nhất).

Hòa (`target − below == over − target`) ⇒ chọn `below` (nhỏ hơn) — đúng luật đề.

**Code nộp LeetCode (Cách 3):**

In [10]:
class SolutionDP:
    def closestCost(self, baseCosts: List[int], toppingCosts: List[int], target: int) -> int:
        can = [False] * (target + 1)
        over = float("inf")
        for b in baseCosts:
            if b <= target:
                can[b] = True
            else:
                over = min(over, b)
        for t in toppingCosts:
            prev = can[:]
            for s in range(target + 1):
                if prev[s]:
                    for k in (1, 2):
                        v = s + k * t
                        if v <= target:
                            can[v] = True
                        else:
                            over = min(over, v)
                            break
        for d in range(target + 1):            # tìm tổng <= target gần nhất
            if can[target - d]:
                below = target - d
                return below if d <= over - target else over
        return over

print([SolutionDP().closestCost(b, t, tg) for b, t, tg, _ in examples])

[10, 17, 8]


**Độ phức tạp Cách 3:** Time `O((n + m) · target)` ≈ `20 · 10⁴`, Space `O(target)`. Không phụ thuộc mũ vào `m` ⇒ tốt khi `m` lớn.

### 7.3 Cách 4 — Meet-in-the-middle + `bisect`

In [11]:
def half_sums(costs):
    S = [0]
    for t in costs:
        S = [s + k * t for s in S for k in range(3)]
    return S

def closest_mitm(baseCosts, toppingCosts, target):
    h = len(toppingCosts) // 2
    A, B = half_sums(toppingCosts[:h]), sorted(set(half_sums(toppingCosts[h:])))
    best = None
    for b in baseCosts:
        for a in A:
            need = target - b - a
            i = bisect.bisect_left(B, need)
            for j in (i - 1, i):
                if 0 <= j < len(B):
                    c = b + a + B[j]
                    if best is None or (abs(c - target), c) < (abs(best - target), best):
                        best = c
    return best

print([closest_mitm(b, t, tg) for b, t, tg, _ in examples])

[10, 17, 8]


**Độ phức tạp Cách 4:** `|A| = 3^{⌊m/2⌋}`, `|B| = 3^{⌈m/2⌉}` ⇒ Time `O(n · 3^{m/2} · log 3^{m/2} + 3^{m/2} log)` ≈ `10 · 243 · 8`; Space `O(3^{m/2})`.

## 8. TRACE từng bước

Trace Cách 2 (DFS) trên **Ví dụ 1** (`base = [1, 7]`, topping `[3, 4]`, target 10), in mỗi lần `best` thay đổi và mỗi nhánh bị cắt:

In [12]:
def trace_dfs(baseCosts, toppingCosts, target):
    best = [min(baseCosts)]; m = len(toppingCosts)
    print("khởi tạo best =", best[0])
    def dfs(i, cost, path):
        if (abs(cost - target), cost) < (abs(best[0] - target), best[0]):
            best[0] = cost; print(f"  best <- {cost}  (chọn {path})")
        if cost >= target and i < m:
            print(f"  cắt tỉa tại {path}: cost={cost} >= target"); return
        if i == m: return
        for k in range(3):
            dfs(i + 1, cost + k * toppingCosts[i], path + [f"{k}x{toppingCosts[i]}"])
    for b in baseCosts:
        print("nền", b); dfs(0, b, [f"nền {b}"])
    print("Kết quả:", best[0]); return best[0]

assert trace_dfs(*examples[0][:3]) == 10

khởi tạo best = 1
nền 1
  best <- 5  (chọn ['nền 1', '0x3', '1x4'])
  best <- 9  (chọn ['nền 1', '0x3', '2x4'])
nền 7
  best <- 10  (chọn ['nền 7', '1x3'])
  cắt tỉa tại ['nền 7', '1x3']: cost=10 >= target
  cắt tỉa tại ['nền 7', '2x3']: cost=13 >= target
Kết quả: 10


Trace Cách 3 (DP) trên **Ví dụ 3** — tập tổng `≤ 9` và `over` sau mỗi topping:

In [13]:
def trace_dp(baseCosts, toppingCosts, target):
    can = [False] * (target + 1); over = float("inf")
    for b in baseCosts:
        if b <= target: can[b] = True
        else: over = min(over, b)
    print("sau nền:", [s for s in range(target + 1) if can[s]], "over =", over)
    for t in toppingCosts:
        prev = can[:]
        for s in range(target + 1):
            if prev[s]:
                for k in (1, 2):
                    v = s + k * t
                    if v <= target: can[v] = True
                    else: over = min(over, v); break
        print(f"sau topping {t}:", [s for s in range(target + 1) if can[s]], "over =", over)
    return closest_dp(baseCosts, toppingCosts, target)

print("Kết quả:", trace_dp(*examples[2][:3]))

sau nền: [3] over = 10
sau topping 2: [3, 5, 7] over = 10
sau topping 5: [3, 5, 7, 8] over = 10
Kết quả: 8


| bước | tập tổng ≤ 9 | over |
|---|---|---|
| sau nền | {3} | 10 |
| sau topping 2 | {3, 5, 7} | 10 |
| sau topping 5 | {3, 5, 7, 8} | 10 |

`below = 8` (cách 1), `over = 10` (cách 1) ⇒ hòa ⇒ **8**.

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — DFS có cắt tỉa (ngắn, nhanh với `m ≤ 10`).

In [14]:
class Solution:
    def closestCost(self, baseCosts: List[int], toppingCosts: List[int], target: int) -> int:
        best = min(baseCosts)
        m = len(toppingCosts)

        def dfs(i: int, cost: int) -> None:
            nonlocal best
            if (abs(cost - target), cost) < (abs(best - target), best):
                best = cost
            if i == m or cost >= target:
                return
            for k in range(3):
                dfs(i + 1, cost + k * toppingCosts[i])

        for b in baseCosts:
            dfs(0, b)
        return best

for b, t, tg, exp in examples:
    got = Solution().closestCost(b, t, tg); print(got, exp); assert got == exp

10 10
17 17
8 8


## 10. Vì sao thuật toán đúng

**Không gian lời giải:** mọi món tráng miệng hợp lệ ứng với một cặp `(nền b, bộ k ∈ {0,1,2}^m)` ⇒ Cách 1 liệt kê đủ, chọn theo khóa `(|c − target|, c)` đúng định nghĩa "gần nhất, hòa lấy nhỏ hơn".

**Cắt tỉa Cách 2 an toàn:** khi `cost ≥ target`, mọi mở rộng có `cost' ≥ cost ≥ target` ⇒ `|cost' − target| ≥ |cost − target|`; nếu bằng nhau thì `cost' ≥ cost` nên không thắng theo luật hòa ⇒ không bỏ sót lời giải tốt hơn. Trạng thái `cost` hiện tại đã được so sánh trước khi cắt.

**Cách 3:** `can` chứa **đúng** mọi tổng `≤ target` đạt được (quy hoạch theo từng topping, dùng `prev` để không lấy một topping quá 2 phần). `over` là tổng nhỏ nhất `> target` (lập luận ở mục 7.2). Đáp án tối ưu hoặc là tổng lớn nhất `≤ target`, hoặc là tổng nhỏ nhất `> target` — mọi tổng khác bị một trong hai "trội hơn". ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(baseCosts)`, `m = len(toppingCosts)`, `T = target`.

| Cách | Time | Space |
|---|---|---|
| 1. `product` | `O(n · 3^m · m)` | `O(m)` |
| 2. DFS cắt tỉa | `O(n · 3^m)` tệ nhất | `O(m)` |
| 3. DP tập tổng | `O((n + m) · T)` | `O(T)` |
| 4. Meet-in-the-middle | `O(n · 3^{m/2} · m)` | `O(3^{m/2})` |

In [15]:
rng = random.Random(1774)
base = [rng.randint(1, 50) for _ in range(10)]; top = [rng.randint(1, 50) for _ in range(10)]; tg = 10 ** 4
for name, f in [("product", closest_bruteforce), ("dfs", Solution().closestCost), ("dp", SolutionDP().closestCost), ("mitm", closest_mitm)]:
    t = time.perf_counter(); r = f(base, top, tg); print(f"{name:<8} -> {r} ({(time.perf_counter()-t)*1000:.0f} ms)")

product  -> 756 (363 ms)
dfs      -> 756 (125 ms)
dp       -> 756 (2 ms)
mitm     -> 756 (1 ms)


Trường hợp trên (giá nhỏ, target lớn) không cắt tỉa được gì — là tệ nhất cho DFS. Ngược lại khi giá lớn hơn target, DFS dừng gần như ngay.

## 12. Bộ test

In [16]:
ALL = {"dfs": closest_dfs, "dp": closest_dp, "mitm": closest_mitm, "submit": Solution().closestCost,
       "SolDP": SolutionDP().closestCost}
def run_all(b, t, tg):
    ref = closest_bruteforce(b, t, tg)
    for k, f in ALL.items():
        assert f(b, t, tg) == ref, (k, b, t, tg)
    return ref

for b, t, tg, exp in examples:
    assert run_all(b, t, tg) == exp
edge = [([5], [1], 1, 5), ([1], [10], 6, 1), ([1], [10], 7, 11), ([10], [1], 3, 10), ([2, 4], [1], 3, 3),
        ([3], [2], 4, 3), ([10000], [10000], 1, 10000)]
for b, t, tg, exp in edge:
    assert run_all(b, t, tg) == exp, (b, t, tg)
rng = random.Random(17740)
for _ in range(400):
    n, m = rng.randint(1, 4), rng.randint(1, 6)
    hi = rng.choice([10, 100, 10 ** 4])
    run_all([rng.randint(1, hi) for _ in range(n)], [rng.randint(1, hi) for _ in range(m)], rng.randint(1, min(hi * 3, 10 ** 4)))
print("Ví dụ, edge case (có hòa), 400 test ngẫu nhiên: OK")

Ví dụ, edge case (có hòa), 400 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Hòa thì lấy giá **lớn** hơn (hoặc giữ ứng viên gặp trước)

In [17]:
def wrong_tie(baseCosts, toppingCosts, target):
    best = None
    for b in baseCosts:
        for ks in product(range(3), repeat=len(toppingCosts)):
            c = b + sum(k * t for k, t in zip(ks, toppingCosts))
            if best is None or abs(c - target) <= abs(best - target):      # SAI: hòa thì ghi đè
                best = c
    return best

b, t, tg, exp = examples[2]
print("sai :", wrong_tie(b, t, tg), "| đúng:", exp)

sai : 10 | đúng: 8


**Vì sao sai:** khi hòa khoảng cách, đề yêu cầu giá **nhỏ hơn**; `<=` giữ ứng viên gặp sau (ở đây 10). **Sửa:** khóa `(abs(c − target), c)`.

### Lỗi 2 — Cho phép không chọn kem nền / chọn nhiều nền

In [18]:
def wrong_no_base(baseCosts, toppingCosts, target):
    items = [(c, 1) for c in baseCosts] + [(c, 2) for c in toppingCosts]   # SAI: nền như topping 0/1 phần
    best = None
    for ks in product(*[range(lim + 1) for _, lim in items]):
        c = sum(k * x for k, (x, _) in zip(ks, items))
        if best is None or (abs(c - target), c) < (abs(best - target), best):
            best = c
    return best

b, t, tg = [8], [1], 2                         # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_no_base(b, t, tg), "| đúng:", Solution().closestCost(b, t, tg))

sai : 2 | đúng: 8


**Vì sao sai:** phải có **đúng một** kem nền. Code sai cho phép chỉ lấy topping (chi phí 2). **Sửa:** duyệt nền ở vòng ngoài, đúng một nền mỗi món.

### Lỗi 3 — Mỗi topping chỉ 0 hoặc 1 phần

In [19]:
def wrong_max_one(baseCosts, toppingCosts, target):
    best = None
    for b in baseCosts:
        for ks in product(range(2), repeat=len(toppingCosts)):            # SAI: range(3)
            c = b + sum(k * t for k, t in zip(ks, toppingCosts))
            if best is None or (abs(c - target), c) < (abs(best - target), best):
                best = c
    return best

b, t, tg, exp = examples[1]
print("sai :", wrong_max_one(b, t, tg), "| đúng:", exp)

sai : 12 | đúng: 17


**Vì sao sai:** Ví dụ 2 cần **2 phần** topping giá 5. **Sửa:** `k ∈ {0, 1, 2}`.

### Lỗi 4 — Cắt tỉa sai: dừng khi `cost > target` **trước** khi so sánh `cost`

In [20]:
def wrong_prune(baseCosts, toppingCosts, target):
    best = [min(baseCosts)]; m = len(toppingCosts)
    def dfs(i, cost):
        if cost > target:                          # SAI: bỏ luôn ứng viên vượt (có thể là tốt nhất)
            return
        if (abs(cost - target), cost) < (abs(best[0] - target), best[0]):
            best[0] = cost
        if i == m: return
        for k in range(3):
            dfs(i + 1, cost + k * toppingCosts[i])
    for b in baseCosts:
        dfs(0, b)
    return best[0]

b, t, tg = [1], [10], 7                        # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_prune(b, t, tg), "| đúng:", Solution().closestCost(b, t, tg), "| brute:", closest_bruteforce(b, t, tg))

sai : 1 | đúng: 11 | brute: 11


**Vì sao sai:** chi phí **vượt** target vẫn có thể là gần nhất (`11` cách 4, còn `1` cách 6). **Sửa:** so sánh `cost` trước, rồi mới cắt các **mở rộng** của nó.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `m ≤ 10`, mỗi phần tử 3 lựa chọn ⇒ `3^m ≈ 6·10⁴` ⇒ vét cạn/DFS là ổn.
- "Gần target nhất, hòa lấy nhỏ hơn" ⇒ khóa `(|c − target|, c)`.
- Chi phí tăng đơn điệu khi thêm ⇒ **cắt tỉa** khi đã `≥ target`.
- Giá trị target nhỏ ⇒ DP tập tổng (knapsack) độc lập với `m`.
- `m` lớn (~30–40) ⇒ **meet-in-the-middle**.

## 15. Biến thể của bài

1. **Mỗi topping tối đa `c_i` phần**: thay `range(3)` bằng `range(c_i + 1)`; DP bounded knapsack.
2. **Chọn đúng `k` loại topping**: thêm chiều đếm vào DP/DFS.
3. **Nhiều nền** (tối đa 2 nền): xét thêm cặp nền.
4. **1755 Closest Subsequence Sum**: meet-in-the-middle với `n ≤ 40`.
5. **416 Partition Equal Subset Sum**, **494 Target Sum**: DP tập tổng.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `product` | Liệt kê mọi tổ hợp | `O(n·3^m·m)` | `O(m)` | Kiểm chứng |
| 2. DFS cắt tỉa | Dừng khi `≥ target` | `O(n·3^m)` tệ nhất | `O(m)` | **Bản nộp** (m nhỏ) |
| 3. DP tập tổng | Tổng `≤ target` + over | `O((n+m)·T)` | `O(T)` | `T` nhỏ, `m` lớn |
| 4. Meet-in-the-middle | Hai nửa + bisect | `O(n·3^{m/2}·m)` | `O(3^{m/2})` | `m` lớn, `T` lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([4], [1], 4)`; (b) `([4], [1], 7)`; (c) `([1], [3], 5)`; (d) `([2, 9], [4], 6)`.

**Bài 2 (tính tay):** Với `base = [5]`, `topping = [2, 3]`, `target = 13`: liệt kê mọi chi phí đạt được, chọn đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `topping_sums(toppingCosts)` trả **tập** mọi tổng topping (mỗi loại 0–2 phần).

**Bài 4 (tự viết lại):** Viết `closest_with_plan(baseCosts, toppingCosts, target)` trả `(chi phí, chỉ số nền, list số phần mỗi topping)`.

**Bài 5 (mini challenge):** Viết `closest_limits(baseCosts, toppingCosts, limits, target)` khi topping `i` lấy tối đa `limits[i]` phần (`0 ≤ limits[i] ≤ 5`), bằng DP tập tổng; đối chiếu vét cạn.

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def topping_sums_todo(toppingCosts):
    S = {0}
    for t in toppingCosts:
        # TODO: S = {s + k*t với s trong S, k trong 0..2}
        pass
    return S

In [22]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def closest_with_plan(baseCosts, toppingCosts, target):
    # TODO: duyệt (chỉ số nền, bộ k) như Cách 1, lưu cả kế hoạch
    pass

def closest_limits(baseCosts, toppingCosts, limits, target):
    # TODO: như Cách 3 nhưng k chạy 1..limits[i]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: đúng một nền. Mức 2: topping 0–2 phần. Mức 3: (a) `4`; (b) chi phí đạt được `{4, 5, 6}` ⇒ `6`; (c) chi phí `{1, 4, 7}`: `4` cách 1, `7` cách 2 ⇒ `4`; (d) `6` (`2 + 4`).

**Bài 2.** Mức 1: tổng topping `{0,2,3,4,5,6,7,8,10}`. Mức 2: chi phí `{5,7,8,9,10,11,12,13,15}`. Mức 3: `13`.

**Bài 3.** Mức 1: tập bắt đầu `{0}`. Mức 2: tập mới từ tập cũ. Mức 3: `S = {s + k*t for s in S for k in range(3)}`.

**Bài 4.** Mức 1: `product(range(3), repeat=m)`. Mức 2: lưu `(c, i, list(ks))` khi tốt hơn. Mức 3: kiểm tra `baseCosts[i] + Σ k·t == c` và `c` là đáp án.

**Bài 5.** Mức 1: `prev = can[:]`. Mức 2: `for k in range(1, limits[i] + 1)`, dừng khi vượt. Mức 3: vét cạn bằng `product(*[range(L + 1) for L in limits])`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2, 3
b1 = [Solution().closestCost(*a) for a in [([4], [1], 4), ([4], [1], 7), ([1], [3], 5), ([2, 9], [4], 6)]]
print("Bài 1:", b1); assert b1 == [4, 6, 4, 6]
def topping_sums_todo(toppingCosts):
    S = {0}
    for t in toppingCosts:
        S = {s + k * t for s in S for k in range(3)}
    return S
costs = sorted(5 + s for s in topping_sums_todo([2, 3]))
print("Bài 2: chi phí =", costs, "-> đáp án", Solution().closestCost([5], [2, 3], 13))
assert costs == [5, 7, 8, 9, 10, 11, 12, 13, 15] and Solution().closestCost([5], [2, 3], 13) == 13

Bài 1: [4, 6, 4, 6]
Bài 2: chi phí = [5, 7, 8, 9, 10, 11, 12, 13, 15] -> đáp án 13


In [24]:
# Bài 4, 5
def closest_with_plan(baseCosts, toppingCosts, target):
    best = None
    for i, b in enumerate(baseCosts):
        for ks in product(range(3), repeat=len(toppingCosts)):
            c = b + sum(k * t for k, t in zip(ks, toppingCosts))
            if best is None or (abs(c - target), c) < (abs(best[0] - target), best[0]):
                best = (c, i, list(ks))
    return best

def closest_limits(baseCosts, toppingCosts, limits, target):
    can = [False] * (target + 1); over = float("inf")
    for b in baseCosts:
        if b <= target: can[b] = True
        else: over = min(over, b)
    for t, L in zip(toppingCosts, limits):
        prev = can[:]
        for s in range(target + 1):
            if prev[s]:
                for k in range(1, L + 1):
                    v = s + k * t
                    if v <= target: can[v] = True
                    else: over = min(over, v); break
    for d in range(target + 1):
        if can[target - d]:
            return target - d if d <= over - target else over
    return over

def limits_brute(baseCosts, toppingCosts, limits, target):
    cands = [b + sum(k * t for k, t in zip(ks, toppingCosts)) for b in baseCosts
             for ks in product(*[range(L + 1) for L in limits])]
    return min(cands, key=lambda c: (abs(c - target), c))

rng = random.Random(4)
for _ in range(300):
    b = [rng.randint(1, 30) for _ in range(rng.randint(1, 3))]; t = [rng.randint(1, 30) for _ in range(rng.randint(1, 4))]
    tg = rng.randint(1, 120)
    c, i, ks = closest_with_plan(b, t, tg)
    assert c == Solution().closestCost(b, t, tg) and b[i] + sum(k * x for k, x in zip(ks, t)) == c
    L = [rng.randint(0, 5) for _ in t]
    assert closest_limits(b, t, L, tg) == limits_brute(b, t, L, tg)
print("Bài 4, 5: OK |", closest_with_plan([2, 3], [4, 5, 100], 18))

Bài 4, 5: OK | (17, 1, [1, 2, 0])


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DFS có cắt tỉa + luật hòa:**

```python
best = min(baseCosts)
def dfs(i, cost):
    nonlocal best
    if (abs(cost - target), cost) < (abs(best - target), best): best = cost
    if i == m or cost >= target: return
    for k in range(3): dfs(i + 1, cost + k * toppingCosts[i])
for b in baseCosts: dfs(0, b)
return best
```

**Checklist:**
- [ ] Đúng **một** kem nền mỗi món?
- [ ] Mỗi topping `0, 1, 2` phần?
- [ ] Hòa khoảng cách ⇒ lấy chi phí **nhỏ hơn**?
- [ ] Cắt tỉa **sau** khi đã so sánh chi phí hiện tại (ứng viên vượt target vẫn hợp lệ)?
- [ ] Đã test: chỉ một nền và giá nền > target, hòa hai phía, cần 2 phần topping?